In [1]:
from functools import cache

from calitp_data_analysis.gcs_pandas import GCSPandas


@cache
def gcs_pandas():
    return GCSPandas()

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [2]:
# need to get ntd ID of all California agencies
from calitp_data_analysis.sql import query_sql

# query agency information, then filter for a single gtfs feed,
# and then count how often each feed key occurs
ca_agencies = query_sql(
    """
    SELECT distinct ntd_id, agency, city, state, agency_voms
    FROM cal-itp-data-infra.mart_ntd.dim_annual_service_agencies
    WHERE state = 'CA' AND report_year = 2024
    """,
    as_df=True,
)

ca_agencies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ntd_id       209 non-null    object 
 1   agency       209 non-null    object 
 2   city         209 non-null    object 
 3   state        209 non-null    object 
 4   agency_voms  209 non-null    float64
dtypes: float64(1), object(4)
memory usage: 8.3+ KB


In [3]:
rev_veh_inv_link = "gs://calitp-analytics-data/data-analyses/ntd/2024 Revenue Vehicle Inventory_250820.xlsx"

rev_veh = gcs_pandas().read_excel(rev_veh_inv_link)
rev_veh["NTD ID"] = rev_veh["NTD ID"].astype("str")

In [4]:
ca_rev_veh = rev_veh[rev_veh["NTD ID"].isin(ca_agencies["ntd_id"].unique().tolist())]

# List of vehicles for CA agencies from the 2024 Revenue Vehicle Inventory

In [5]:
ca_rev_veh = ca_rev_veh.groupby(
    [
        "NTD ID",
        "Agency Name",
    ]
).agg({
    "Total Fleet Vehicles": "sum",
    "ModeTOS Vehicles Operated in Maximum Service":"max"
}).reset_index()

## difference between VOMs and Rev Vehicle Inv.

In [6]:
diff_voms_ref_veh = ca_rev_veh.merge(
    ca_agencies,
    left_on="NTD ID",
    right_on="ntd_id",
    how="inner",
    indicator=True
)

In [9]:
diff_voms_ref_veh["voms_diff"] = diff_voms_ref_veh["ModeTOS Vehicles Operated in Maximum Service"] - diff_voms_ref_veh["agency_voms"]

In [13]:
diff_voms_ref_veh["rev_veh_voms_diff"] = diff_voms_ref_veh["Total Fleet Vehicles"] - diff_voms_ref_veh["ModeTOS Vehicles Operated in Maximum Service"]

In [16]:
diff_voms_ref_veh.describe()

,Total Fleet Vehicles,ModeTOS Vehicles Operated in Maximum Service,agency_voms,voms_diff,rev_veh_voms_diff
count,209.000000,209.000000,209.000000,209.000000,209.000000
mean,133.033493,54.401914,83.430622,-29.028708,78.631579
std,390.845120,147.942430,244.023419,113.089903,260.334037
min,1.000000,1.000000,1.000000,-1246.000000,-1.000000
25%,10.000000,4.000000,5.000000,-12.000000,4.000000
50%,23.000000,10.000000,15.000000,-4.000000,13.000000
75%,83.000000,29.000000,42.000000,0.000000,47.000000
max,4415.000000,1476.000000,2722.000000,0.000000,2939.000000


In [17]:
diff_voms_ref_veh

,NTD ID,Agency Name,Total Fleet Vehicles,ModeTOS Vehicles Operated in Maximum Service,ntd_id,agency,city,state,agency_voms,_merge,voms_diff,rev_veh_voms_diff
0,90003,San Francisco Bay Area Rapid Transit District,785,566.0,90003,"San Francisco Bay Area Rapid Transit District,...",Oakland,CA,582.0,both,-16.0,219.0
1,90004,Golden Empire Transit District,160,51.0,90004,Golden Empire Transit District,Bakersfield,CA,93.0,both,-42.0,109.0
2,90006,Santa Cruz Metropolitan Transit District,147,58.0,90006,Santa Cruz Metropolitan Transit District,Santa Cruz,CA,97.0,both,-39.0,89.0
3,90008,City of Santa Monica,239,124.0,90008,"City of Santa Monica, dba: Big Blue Bus",Santa Monica,CA,162.0,both,-38.0,115.0
4,90009,San Mateo County Transit District,468,178.0,90009,"San Mateo County Transit District, dba: SamTrans",San Carlos,CA,361.0,both,-183.0,290.0
5,90010,City of Torrance,104,38.0,90010,"City of Torrance, dba: Torrance Transit System",Torrance,CA,74.0,both,-36.0,66.0
6,90012,San Joaquin Regional Transit District,178,74.0,90012,"San Joaquin Regional Transit District, dba: Sa...",Stockton,CA,105.0,both,-31.0,104.0
7,90013,Santa Clara Valley Transportation Authority,818,341.0,90013,"Santa Clara Valley Transportation Authority, d...",San Jose,CA,520.0,both,-179.0,477.0
8,90014,Alameda-Contra Costa Transit District,851,371.0,90014,"Alameda-Contra Costa Transit District, dba: AC...",Oakland,CA,619.0,both,-248.0,480.0
9,90015,City and County of San Francisco,1366,364.0,90015,"City and County of San Francisco, dba: San Fra...",San Francisco,CA,777.0,both,-413.0,1002.0


# Count of routes